In [0]:
from pyspark.sql import functions as F

T_trans = spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
T_locations = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location")
T_date = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_date")

In [0]:
from pyspark.sql import functions as F

T_trans = spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
T_locations = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location")
T_date = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_date")

df_all_trans = (
    T_trans
    .join(T_locations.select("LocationID", "LC_Location_Type_Description", "LC_Trade_Brand_Description"),
          T_trans.LocationId == T_locations.LocationID, how="left")
    .join(T_date.select("dateid", "datefull"),
          T_trans.VisitDateId == T_date.dateid, how="left")
    .filter(F.col("datefull").between(F.add_months(F.current_date(), -12), '2026-05-18'))
    .filter(F.col("LC_Location_Type_Description").isin("Cineplex Theatre", "Restaurant/Store/Games"))
    .filter(F.col("SCENEFLAG") == "Yes")
    .filter(F.col("CDE_ID").isNotNull())
    .withColumn("visit_key", F.concat_ws("|", "visitdateid", T_trans.LocationId, "CDE_ID"))
    .withColumn("Month", F.month("datefull"))
    .withColumn("Year", F.year("datefull"))
    .select(
        "CDE_ID",
        "datefull",
        "Month",
        "Year",
        "visit_key",
        T_trans.LocationId.alias("LocationId"),
        "LC_Location_Type_Description",
        "LC_Trade_Brand_Description",
        "TicketFLAG",
        "ConcessionFLAG",
        "CineClubFLAG",
        "Quantity",
        "Net_Revenue",
        "Net_Spend",
        "CineClub_Discount_Type_ID",
    )
)

In [0]:
from datetime import date, timedelta

yesterday_id = (date.today() - timedelta(days=2)).strftime("%Y%m%d")

T_cc_snapshot = spark.table("cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot")

cineclub_members = (
    T_cc_snapshot
    .filter(F.col("snapshot_date_id") == yesterday_id)
    .filter(F.col("renewal_status").like("Active%"))
    .select(F.col("cde_id").alias("CDE_ID"))
    .distinct()
)

In [0]:
cineclub_members.display()

In [0]:
# 2. Scene-only (excluding active CC members) — spend per member by brand
scene_overview = (
    df_all_trans
    .join(cineclub_members, on="CDE_ID", how="left_anti")
    .groupBy("LC_Trade_Brand_Description")
    .agg(
        F.countDistinct("CDE_ID").alias("members"),
        F.countDistinct("visit_key").alias("visits"),
        F.sum("Net_Revenue").alias("total_Revenue"),
    )
    .withColumn("Revenue_per_member", F.col("total_Revenue") / F.col("members"))
    .withColumn("visits_per_member", F.col("visits") / F.col("members"))
)

In [0]:
scene_overview.display()

In [0]:
# 3. Active CC members — revenue per member by brand
cc_overview = (
    df_all_trans
    .join(cineclub_members, on="CDE_ID", how="inner")
    .groupBy("LC_Trade_Brand_Description")
    .agg(
        F.countDistinct("CDE_ID").alias("members"),
        F.countDistinct("visit_key").alias("visits"),
        F.sum("Net_Revenue").alias("total_Revenue"),
    )
    .withColumn("Revenue_per_member", F.col("total_Revenue") / F.col("members"))
    .withColumn("visits_per_member", F.col("visits") / F.col("members"))
)

In [0]:
cc_overview.display()

In [0]:
# 4. Active CC members — revenue per member by brand × discount type
cc_overview_by_discount = (
    df_all_trans
    .join(cineclub_members, on="CDE_ID", how="inner")
    .groupBy("LC_Trade_Brand_Description", "CineClub_Discount_Type_ID")
    .agg(
        F.countDistinct("CDE_ID").alias("members"),
        F.countDistinct("visit_key").alias("visits"),
        F.sum("Net_Revenue").alias("total_Revenue"),
    )
    .withColumn("Revenue_per_member", F.col("total_Revenue") / F.col("members"))
    .withColumn("visits_per_member", F.col("visits") / F.col("members"))
    .orderBy("LC_Trade_Brand_Description", "CineClub_Discount_Type_ID")
)

In [0]:
cc_overview_by_discount.display()

In [0]:
# 5. Tag CC members by LBE visitation, compare revenue per visit
cc_lbe_visitors = (
    df_all_trans
    .join(cineclub_members, on="CDE_ID", how="inner")
    .filter(F.col("LC_Trade_Brand_Description").isin("Rec Room", "Playdium"))
    .select("CDE_ID").distinct()
    .withColumn("visited_lbe", F.lit(1))
)

cc_lbe_comparison = (
    df_all_trans
    .join(cineclub_members, on="CDE_ID", how="inner")
    .join(cc_lbe_visitors, on="CDE_ID", how="left")
    .fillna(0, subset=["visited_lbe"])
    .groupBy("visited_lbe", "LC_Trade_Brand_Description")
    .agg(
        F.countDistinct("CDE_ID").alias("members"),
        F.countDistinct("visit_key").alias("visits"),
        F.sum("Net_Revenue").alias("total_Revenue"),
    )
    .withColumn("Revenue_per_visit", F.col("total_Revenue") / F.col("visits"))
    .orderBy("visited_lbe", "LC_Trade_Brand_Description")
)

In [0]:
cc_lbe_comparison.display()

In [0]:
segment_sizes = (
    df_all_trans
    .join(cineclub_members, on="CDE_ID", how="inner")
    .join(cc_lbe_visitors, on="CDE_ID", how="left")
    .fillna(0, subset=["visited_lbe"])
    .groupBy("visited_lbe")
    .agg(F.countDistinct("CDE_ID").alias("unique_members"))
)

In [0]:
segment_sizes.display()

In [0]:
import pandas as pd
from pyspark.sql.window import Window

# A. CC members who have NOT visited LBE in window
cc_non_lbe_members = (
    cineclub_members
    .join(cc_lbe_visitors, on="CDE_ID", how="left_anti")
)

# B. Theatre -> closest LBE lookup (from Databricks table)
lbe_dist = (
    spark.table("data_exploration_hub.marketing_hub.lbe_location_distance")
    .filter(F.col("Theater_ID").isNotNull())
    .filter(~F.col("LBE_Name").contains("Vaughan Mills"))   # drop unbuilt LBE
)

w_theatre = Window.partitionBy("Theater_ID").orderBy(F.col("Distance_KM").asc())

theatre_to_lbe = (
    lbe_dist
    .withColumn("rn", F.row_number().over(w_theatre))
    .filter(F.col("rn") == 1)
    .select(
        F.col("Theater_ID").cast("int").alias("LocationId"),
        F.col("LBE_Name").alias("closest_lbe"),
        F.col("Distance_KM").alias("distance_km"),
        F.col("Distance_Band").alias("distance_band"),
    )
)
# C. Each non-LBE CC member's primary (most-visited) theatre
member_theatre_visits = (
    df_all_trans
    .join(cc_non_lbe_members, on="CDE_ID", how="inner")
    .filter(F.col("LC_Location_Type_Description") == "Cineplex Theatre")
    .groupBy("CDE_ID", "LocationId")
    .agg(F.countDistinct("visit_key").alias("theatre_visits"))
)

w_member = Window.partitionBy("CDE_ID").orderBy(
    F.col("theatre_visits").desc(),
    F.col("LocationId").asc(),
)

member_primary_theatre = (
    member_theatre_visits
    .withColumn("rn", F.row_number().over(w_member))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# D. Member -> closest LBE -> aggregate
lbe_proximity_summary = (
    member_primary_theatre
    .join(theatre_to_lbe, on="LocationId", how="left")
    .withColumn("closest_lbe", F.coalesce(F.col("closest_lbe"), F.lit("No LBE within 50 km")))
    .withColumn("distance_band", F.coalesce(F.col("distance_band"), F.lit("50+ km")))
    .groupBy("closest_lbe", "distance_band")
    .agg(F.countDistinct("CDE_ID").alias("members"))
    .orderBy("closest_lbe", "distance_band")
)

In [0]:
lbe_proximity_summary.display()

In [0]:
# 1. Does member_primary_theatre have duplicates per CDE_ID?
print("rows:", member_primary_theatre.count())
print("distinct CDE_IDs:", member_primary_theatre.select("CDE_ID").distinct().count())

# 2. Does theatre_to_lbe have duplicates per LocationId?
theatre_to_lbe.groupBy("LocationId").count().filter(F.col("count") > 1).show()